# 06 — Data access points

The `data_access` extension connects an instrument to collections exposed by platforms such as Google Earth Engine, Microsoft Planetary Computer, the Copernicus Data Space Ecosystem, and EOPF Sentinel Zarr Samples. This tutorial uses Sentinel-2A MSI to discover the available providers and their collection identifiers.

In [ ]:
import pandas as pd
import xeo

instrument = xeo.instruments.MSI_S2A
instrument

## Discover available providers

Check for the extension before reading it. Provider names are the keys of the `data_access` mapping.

In [ ]:
print("Available extensions:", instrument.extension_names)

data_access = instrument.extensions.get("data_access", {})
providers = list(data_access)
print("Data access providers:", providers)

## Get the primary collection ID for each provider

Every provider can identify one access point as `primary`. The collection ID is stored in that access point's `collection` field.

In [ ]:
primary_collection_ids = {
    provider: provider_metadata["primary"]["collection"]
    for provider, provider_metadata in data_access.items()
    if "primary" in provider_metadata
}
primary_collection_ids

## List every available access point

Providers may expose several processing levels, such as bottom-of-atmosphere (`boa`) and top-of-atmosphere (`toa`) collections. The helper below flattens the extension into a DataFrame while ignoring provider-level fields such as `stac_endpoint`.

In [ ]:
def data_access_table(instrument):
    rows = []
    data_access = instrument.extensions.get("data_access", {})

    for provider, provider_metadata in data_access.items():
        stac_endpoint = provider_metadata.get("stac_endpoint")
        for access_point, access_metadata in provider_metadata.items():
            if not isinstance(access_metadata, dict):
                continue
            if "collection" not in access_metadata:
                continue

            rows.append(
                {
                    "provider": provider,
                    "access_point": access_point,
                    "collection": access_metadata["collection"],
                    "stac_endpoint": stac_endpoint,
                    "docs": access_metadata.get("docs"),
                }
            )

    return pd.DataFrame(rows)


access_points = data_access_table(instrument)
access_points

The resulting table makes it easy to select collection IDs by provider or processing level.

In [ ]:
access_points.loc[
    access_points["access_point"].isin(["boa", "toa"]),
    ["provider", "access_point", "collection"],
].sort_values(["provider", "access_point"])

## Get STAC endpoints

STAC-based providers include a provider-level endpoint. Google Earth Engine uses its own collection identifiers and therefore has no `stac_endpoint` here.

In [ ]:
stac_endpoints = {
    provider: provider_metadata["stac_endpoint"]
    for provider, provider_metadata in data_access.items()
    if "stac_endpoint" in provider_metadata
}
stac_endpoints

## Reuse the workflow with other instruments

First discover which instruments provide the extension, then pass any of them to `data_access_table()`.

In [ ]:
instruments_with_data_access = [
    item.id
    for item in xeo.instruments.values()
    if "data_access" in item.extension_names
]
print(instruments_with_data_access)

data_access_table(xeo.instruments.ASTER)